In [ ]:
load_ext jupyter_black

In [ ]:
import numpy as np
import pickle
import os
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import f_oneway
import pingouin as pg

In [ ]:
demographics = {
    "prism": [
        "age",
        "gender",
        "employment_status",
        "education",
        "marital_status",
        "english_proficiency",
        "religion",
        "ethnicity",
        "birth_region",
        "reside_region",
        "lm_familiarity",
    ],
    "cad_en": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
}

domains = ["salary", "benefits", "political"]  # "legal",  # "medical",

In [ ]:
def get_answers(model):
    questions = pd.read_pickle(f"data/{model}_questions.gz")
    questions_correct_answers = dict(zip(questions.q_id, questions.correct_answer))
    questions_baseline_answers = questions[["q_id", "baseline_answer", "domain"]]

    questions_baseline_answers = questions_baseline_answers.loc[
        ~questions_baseline_answers["q_id"].isin([f"q_{i}" for i in range(61)])
    ].reset_index(drop=True)

    domain_qid_map = {
        domain: questions.loc[questions["domain"] == domain, "q_id"].tolist()
        for domain in domains
    }

    questions_baseline_answers.loc[
        questions_baseline_answers["domain"] == "salary",
        "baseline_answer",
    ] = (
        questions_baseline_answers.loc[
            questions_baseline_answers["domain"] == "salary",
            "baseline_answer",
        ]
        .str.replace(",", "")
        .str.extract(r"^[^\d]*(\d+)", expand=False)
        .astype(float)
    )

    questions_baseline_answers = pd.Series(
        questions_baseline_answers.baseline_answer.values,
        index=questions_baseline_answers.q_id,
    ).to_dict()
    return domain_qid_map, questions_correct_answers, questions_baseline_answers

In [ ]:
def get_stats(df_dataset, x, y):
    group_mappings = {}
    i = 0
    total_values = []
    for group in df_dataset[x].unique():
        values = df_dataset[df_dataset[x] == group][y].tolist()
        if len(values) == 0:
            continue
        total_values.append(values)
        group_mappings[i] = group
        i += 1
    anova = f_oneway(*total_values, axis=None)
    # tukey = tukey_hsd(*total_values)
    return anova  # tukey, group_mappings

In [ ]:
def get_plots(
    df_dataset,
    x,
    y,
    anova,
    ax=None,
    baseline=None,
    save="",
    anova_label=False,
):
    plt.rcParams.update({"font.size": 22})
    df_dataset = df_dataset.sort_values(by=x)
    label = None
    if anova_label:
        label = "before" if before else "after"
        if anova.pvalue < 0.01:
            label += " *"
    ax = sns.pointplot(
        data=df_dataset,
        ax=ax,
        x=x,
        y=y,
        label=label,
    )
    if anova.pvalue < 0.01 and not anova_label:
        plt.title(x + "*")
    else:
        plt.title(x)

    ax.set_xlabel("")
    label_dict = {
        "salary": "Average salary",
        "medical": "Encouraged to seek medical\nattention",
        "benefits": "Is told they are eligible for\nbenefits",
        "legal": "Receives legally advantageous\nadvice",
        "political": "Receives politically liberal\nanswers",
    }

    ax.set_ylabel(label_dict[y])

    for lbl in ax.get_xticklabels():
        lbl.set_rotation(30)
        lbl.set_horizontalalignment("right")

    if baseline:
        ax.axhline(
            y=baseline,
            color="orange",
            linewidth=4,
        )
    if save:
        plt.savefig(f"figures_before/{save}.pdf", bbox_inches="tight")
        plt.show()
    else:
        return ax

In [ ]:
overview_df = {
    "model": [],
    "dataset": [],
    "domain": [],
    "demo": [],
    "diff": [],
    "F": [],
}
overview_df_debias = {
    "model": [],
    "dataset": [],
    "domain": [],
    "demo": [],
    "diff": [],
    "F": [],
}

for model in [
    "Llama-3.1-8B-Instruct",
    # "Qwen3.6-27B"
]:
    for dataset in [
        "prism",
    ]:
        domain_qid_map, questions_correct_answers, questions_baseline_answers = (
            get_answers(model)
        )
        if model == "Llama-3.1-8B-Instruct":
            df = pd.read_pickle(f"behavior/{model}_{dataset}_answers.gz")
            for c in [
                qid for d in domains for qid in domain_qid_map[d] if d != "salary"
            ]:
                df[c] = 1 * (df[c].str.lower() == questions_correct_answers[c])
            for c in domain_qid_map["salary"]:
                df[c] = (
                    df[c]
                    .str.replace(",", "")
                    .str.extract(r"^[^\d]*(\d+)")
                    .astype(float)
                )

            for domain in domains:
                df[domain] = df[[qid for qid in domain_qid_map[domain]]].mean(axis=1)
                if domain != "salary":
                    df[domain] = df[domain] * 100
            df = df.drop(
                columns=[f"q_{i}" for i in range(50)]
                + ["q_59", "q_60"]
                + [f"q_{i}" for i in range(61, 211)],
                errors="ignore",
            )
        for domain in domains:
            df_debias = pd.read_pickle(
                f"behavior/{model}_{dataset}_{domain}_debias_answers.gz"
            )
            if domain != "salary":
                for c in domain_qid_map[domain]:
                    df_debias[c] = 1 * (
                        df_debias[c].str.lower() == questions_correct_answers[c]
                    )
            else:
                for c in domain_qid_map["salary"]:
                    df_debias[c] = (
                        df_debias[c]
                        .str.replace(",", "")
                        .str.extract(r"^[^\d]*(\d+)")
                        .astype(float)
                    )

            df_debias[domain] = df_debias[[qid for qid in domain_qid_map[domain]]].mean(
                axis=1
            )
            if domain != "salary":
                df_debias[domain] = df_debias[domain] * 100

            df_debias = df_debias.drop(
                columns=[f"q_{i}" for i in range(50)]
                + ["q_59", "q_60"]
                + [f"q_{i}" for i in range(61, 211)],
                errors="ignore",
            )

            if model != "Llama-3.1-8B-Instruct":
                df = pd.read_pickle(f"behavior/{model}_{dataset}_{domain}_answers.gz")
                if domain != "salary":
                    for c in domain_qid_map[domain]:
                        df[c] = 1 * (df[c].str.lower() == questions_correct_answers[c])
                else:
                    for c in domain_qid_map["salary"]:
                        df[c] = (
                            df[c]
                            .str.replace(",", "")
                            .str.extract(r"^[^\d]*(\d+)")
                            .astype(float)
                        )

                df[domain] = df[[qid for qid in domain_qid_map[domain]]].mean(axis=1)
                if domain != "salary":
                    df[domain] = df[domain] * 100

                df = df.drop(
                    columns=[f"q_{i}" for i in range(50)]
                    + ["q_59", "q_60"]
                    + [f"q_{i}" for i in range(61, 211)],
                    errors="ignore",
                )

            filtered_df = df.loc[~df[domain].isna()]
            filtered_df_debias = df_debias.loc[~df[domain].isna()]

            for demo in demographics[dataset]:
                demo_filtered_df = filtered_df.loc[
                    ~(filtered_df[demo].isna())
                    & (filtered_df[demo] != "Prefer not to say")
                    & (filtered_df[demo] != "Other")
                    & (filtered_df[demo] != "Unknown")
                ]
                anova = get_stats(demo_filtered_df, demo, domain)
                demo_filtered_df_debias = filtered_df_debias.loc[
                    ~(filtered_df_debias[demo].isna())
                    & (filtered_df_debias[demo] != "Prefer not to say")
                    & (filtered_df_debias[demo] != "Other")
                    & (filtered_df_debias[demo] != "Unknown")
                ]
                anova_debias = get_stats(demo_filtered_df_debias, demo, domain)
                overview_df["model"].append(model)
                overview_df["dataset"].append(dataset)
                overview_df["domain"].append(domain)
                overview_df["demo"].append(demo)
                overview_df["diff"].append(anova.pvalue < 0.01)
                overview_df["F"].append(anova.statistic)
                overview_df_debias["model"].append(model)
                overview_df_debias["dataset"].append(dataset)
                overview_df_debias["domain"].append(domain)
                overview_df_debias["demo"].append(demo)
                overview_df_debias["diff"].append(anova_debias.pvalue < 0.01)
                overview_df_debias["F"].append(anova_debias.statistic)
                # get_plots(
                #     demo_filtered_df,
                #     demo,
                #     domain,
                #     anova,
                #     baseline=baseline,
                #     save=f"{model}_{dataset}_{domain}_{demo}{'_diff' if anova.pvalue < 0.01 else ''}",
                # )

In [ ]:
overview_df = pd.DataFrame(overview_df)
overview_df_debias = pd.DataFrame(overview_df_debias)

In [ ]:
(
    (
        overview_df.groupby("model")["diff"].sum(),
        overview_df.groupby("model")["diff"].count(),
    ),
    (
        overview_df_debias.groupby("model")["diff"].sum(),
        overview_df_debias.groupby("model")["diff"].count(),
    ),
)

In [ ]:
(
    overview_df.groupby("domain")["diff"].sum(),
    overview_df_debias.groupby("domain")["diff"].sum(),
)

In [ ]:
(
    overview_df.groupby(["domain", "model"])["diff"].sum(),
    overview_df_debias.groupby(["domain", "model"])["diff"].sum(),
)

In [ ]:
(
    overview_df.groupby("demo")["diff"].mean().sort_values(),
    overview_df_debias.groupby("demo")["diff"].mean().sort_values(),
)

In [ ]:
(
    overview_df.groupby(["demo", "domain"])["diff"].mean(),
    overview_df_debias.groupby(["demo", "domain"])["diff"].mean(),
)

In [ ]:
(
    (
        overview_df.groupby("model")["F"].sum(),
        overview_df.groupby("model")["F"].count(),
    ),
    (
        overview_df_debias.groupby("model")["F"].sum(),
        overview_df_debias.groupby("model")["F"].count(),
    ),
)

In [ ]:
(
    overview_df.groupby("domain")["F"].sum(),
    overview_df_debias.groupby("domain")["F"].sum(),
)

In [ ]:
(
    overview_df.groupby(["domain", "model"])["F"].sum(),
    overview_df_debias.groupby(["domain", "model"])["F"].sum(),
)